In [1]:
import os

In [2]:
%pwd

'c:\\Users\\palsa\\OneDrive\\Desktop\\End-to-End-Chest-cancer-classification-using-mlflow\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\palsa\\OneDrive\\Desktop\\End-to-End-Chest-cancer-classification-using-mlflow'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

        

    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "Chest-CT-Scan-data")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [8]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [9]:
import math
import tensorflow as tf
from pathlib import Path
from cnnClassifier.entity.config_entity import TrainingConfig
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    
    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path,
            compile=False
        )
        self.model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path, include_optimizer=False)



    
    def train(self):
        self.steps_per_epoch = math.ceil(
            self.train_generator.samples/
            self.train_generator.batch_size
        )
        self.validation_steps = math.ceil(
            self.valid_generator.samples /
            self.valid_generator.batch_size
            )
        self.validation_steps = math.ceil(
            self.validation_steps /
            self.valid_generator.batch_size
            )

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )



In [10]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e

[2026-05-08 16:07:15,030: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-08 16:07:15,032: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-08 16:07:15,033: INFO: common: created directory at: artifacts]
[2026-05-08 16:07:15,035: INFO: common: created directory at: artifacts\training]
[2026-05-08 16:07:15,288: WARNING: config: TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.]
Found 68 images belonging to 2 classes.
Found 275 images belonging to 2 classes.
18/18 ━━━━━━━━━━━━━━━━━━━━ 45s 3s/step - accuracy: 0.5055 - loss: 14.6298 - val_accuracy: 0.0000e+00 - val_loss: 57.0468
[2026-05-08 16:08:01,148: WARNING: saving_api: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format,

In [1]:
config = ConfigurationManager()

prepare_base_model_config = config.get_prepare_base_model_config()

prepare_base_model = PrepareBaseModel(
    config=prepare_base_model_config
)

prepare_base_model.get_base_model()
prepare_base_model.update_base_model()

NameError: name 'ConfigurationManager' is not defined

In [2]:
from cnnClassifier.config.configuration import ConfigurationManager

In [3]:
from cnnClassifier.components.prepare_base_model import PrepareBaseModel
from cnnClassifier.config.configuration import ConfigurationManager

config = ConfigurationManager()

prepare_base_model_config = config.get_prepare_base_model_config()

prepare_base_model = PrepareBaseModel(
    config=prepare_base_model_config
)

prepare_base_model.get_base_model()
prepare_base_model.update_base_model()

FileNotFoundError: [Errno 2] No such file or directory: 'config\\config.yaml'

In [4]:
import os
print(os.getcwd())

c:\Users\palsa\OneDrive\Desktop\End-to-End-Chest-cancer-classification-using-mlflow\research


In [6]:
import os

os.chdir(r"C:\Users\palsa\OneDrive\Desktop\End-to-End-Chest-cancer-classification-using-mlflow")

print(os.getcwd())

C:\Users\palsa\OneDrive\Desktop\End-to-End-Chest-cancer-classification-using-mlflow


In [7]:
import os
print(os.path.exists("config/config.yaml"))

True


In [8]:
from cnnClassifier.components.prepare_base_model import PrepareBaseModel
from cnnClassifier.config.configuration import ConfigurationManager

config = ConfigurationManager()

prepare_base_model_config = config.get_prepare_base_model_config()

prepare_base_model = PrepareBaseModel(
    config=prepare_base_model_config
)

prepare_base_model.get_base_model()
prepare_base_model.update_base_model()

[2026-05-08 15:58:52,929: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-08 15:58:52,945: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-08 15:58:52,947: INFO: common: created directory at: artifacts]
[2026-05-08 15:58:52,949: INFO: common: created directory at: artifacts/prepare_base_model]
[2026-05-08 15:58:53,409: WARNING: saving_api: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]
[2026-05-08 15:58:53,545: WARNING: config: TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.]


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │        50,178 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,764,866 (56.32 MB)

 Trainable params: 50,178 (196.01 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

[2026-05-08 15:58:53,571: WARNING: saving_api: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]


In [11]:
from cnnClassifier.components.training import Training

training_config = config.get_training_config()

training = Training(config=training_config)

training.get_base_model()
training.train_valid_generator()
training.train()

ModuleNotFoundError: No module named 'cnnClassifier.components.training'

In [12]:
from cnnClassifier.components.model_trainer import Training

In [14]:
from cnnClassifier.components.model_trainer import Training

training_config = config.get_training_config()

training = Training(config=training_config)

training.get_base_model()
training.train_valid_generator()
training.train()

[2026-05-08 16:06:12,396: INFO: common: created directory at: artifacts\training]
[2026-05-08 16:06:12,548: WARNING: legacy_h5_format: No training configuration found in the save file, so the model was *not* compiled. Compile it manually.]
Found 68 images belonging to 2 classes.
Found 275 images belonging to 2 classes.


ValueError: You must call `compile()` before using the model.